# 第 5 周练习 —— Ruby on Rails 文档 RAG

## 练习目标
基于本地 `rorkb` 知识库（从共享 Google Drive 下载的 Ruby on Rails 文档），搭建一条完整的 **RAG** 流水线：

1. 读取 Markdown 文档
2. 用 LLM **结构化分块**（headline / summary / original_text）
3. 写入 **Chroma** 向量库（`text-embedding-3-large`）
4. **查询改写 + 重排序（rerank）** 后再生成答案
5. 用 **Gradio** 提供双栏聊天界面（对话 + 检索上下文）

## 和本课 Week 5 的关系
| 概念 | 本练习里的位置 |
|------|----------------|
| Document / Chunk | `Result` / `Chunk` / `Chunks` |
| Embedding + Vector DB | `create_embeddings` + Chroma `PersistentClient` |
| Advanced RAG | `rewrite_query`、`rerank`、`fetch_context` |
| Chat UI | Gradio `Blocks` |

## 怎么跑
1. 准备好 `.env` 中的 OpenAI 密钥；本地放好 `rorkb/` 文档目录
2. 按单元格顺序运行：导入 → 数据模型 → 取文档 → LLM 分块 → 建库 → RAG → Gradio
3. 分块步骤会调用 LLM，文档多时较慢，属预期行为


In [ ]:
# ========== 导入依赖与全局常量 ==========

# Path：用面向对象方式表示知识库目录
from pathlib import Path
# OpenAI：官方客户端，后面算 embedding
from openai import OpenAI
# load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv
# pydantic：用 BaseModel/Field 约束 LLM 结构化输出（chunks / 排序）
from pydantic import BaseModel, Field
# PersistentClient：Chroma 持久化向量库客户端
from chromadb import PersistentClient
# tqdm：长循环进度条（批量分块时好看进度）
from tqdm import tqdm
# litellm.completion：统一调用聊天模型（可挂 response_format）
from litellm import completion
# numpy：数值数组（本练习导入保留，可视化相关）
import numpy as np
# TSNE：高维降维（本练习导入保留）
from sklearn.manifold import TSNE
# plotly：交互图（本练习导入保留）
import plotly.graph_objects as go

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)

# 聊天/分块用的模型 id（字符串勿改）
MODEL = "gpt-4.1-nano"
# Chroma 持久化目录名
DB_NAME = "preprocessed_db"
# 本地知识库根目录（Ruby on Rails 文档）
KNOWLEDGE_BASE_PATH = Path("rorkb")
# 估算「大概切几块」时用的平均块长
AVERAGE_CHUNK_SIZE = 500

# Chroma collection 名称
collection_name = "docs"
# Embedding 模型 id（大模型向量，检索更准、更贵）
embedding_model = "text-embedding-3-large"
# 默认读取环境变量里的 OPENAI_API_KEY
openai = OpenAI()


In [ ]:
# ========== 数据模型：Result / Chunk / Chunks（给 LLM 结构化输出用）==========

# 类似 LangChain Document：正文 + 元数据
class Result(BaseModel):
    # 检索/展示用的文本内容
    page_content: str
    # 来源路径、文档类型等
    metadata: dict

# 单个语义块：标题 + 摘要 + 原文（Field.description 会进 schema，供模型理解字段）
class Chunk(BaseModel):
    # 短标题：容易被查询命中的关键词短语（description 英文保持原样）
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    # 几句摘要：回答常见问题时够用
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    # 原文片段：要求一字不改，保证可溯源
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    # 把 Chunk 转成 Result：拼接三部分正文，并带上文档 source/type
    def as_result(self, document):
        # metadata 来自原始 document 字典
        metadata = {"source": document["source"], "type": document["type"]}
        # page_content = 标题 + 摘要 + 原文，中间空行分隔
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)

# 一整篇文档对应的块列表（response_format=Chunks 时用）
class Chunks(BaseModel):
    chunks: list[Chunk]


## 第 1 步：从 `rorkb` 文件夹读取文档

遍历知识库子目录，把每个 `.md` 读成 `{type, source, text}` 字典，供后续 LLM 分块。


In [ ]:
# ========== 自制 DirectoryLoader：递归读取 rorkb 下所有 Markdown ==========

def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    # 累积所有文档字典
    documents = []

    # 一级子目录名当作 type（例如 guides / api 等）
    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        # rglob：递归匹配所有 .md
        for file in folder.rglob("*.md"):
            # utf-8 读全文
            with open(file, "r", encoding="utf-8") as f:
                # source 用 posix 路径字符串，跨平台稳定
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    # 报告加载数量
    print(f"Loaded {len(documents)} documents")
    return documents

# 立即执行：后面分块要用 documents
documents = fetch_documents()


## 第 2 步：借助 LLM 做结构化分块

不是简单按字符硬切，而是让模型产出 `headline / summary / original_text`，更利于检索与回答。


In [ ]:
# ========== 构造「请把文档切成重叠块」的长提示词 ==========

# document：含 type / source / text 的字典
def make_prompt(document):
    # 粗估块数：全文长度 ÷ 平均块长，至少 1
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    # f-string 多行提示：指令与文档正文插值（英文 prompt 保持原样，勿翻译）
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called RoR Inc.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

# 用第一篇文档预览提示词长什么样（调试用）
print(make_prompt(documents[0]))


In [ ]:
# ========== 调用 LLM 生成 Chunks，并批量处理全部文档 ==========

# 把提示包成单条 user message
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

# 单文档：completion + 解析 JSON → Result 列表
def process_document(document):
    # 组装 messages
    messages = make_messages(document)
    # response_format=Chunks：要求模型按 pydantic schema 返回
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    # 取出 JSON 字符串
    reply = response.choices[0].message.content
    # 校验并解析为 Chunk 列表
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    # 每个 Chunk 转成带 metadata 的 Result
    return [chunk.as_result(document) for chunk in doc_as_chunks]

# 多文档：tqdm 进度条，extend 合并
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

# 跑完全部文档的 LLM 分块（耗时步骤）
chunks = create_chunks(documents)


In [ ]:
# 打印分块总数，确认上一格成功产出 chunks
print(len(chunks))


## 步骤 3：把块写成向量嵌入并入库

清空旧 collection（若存在）→ 批量 embedding → `collection.add` 写入 Chroma。


In [ ]:
# ========== 创建/重建 Chroma collection，写入 embeddings ==========

def create_embeddings(chunks):
    # 打开（或创建）持久化 Chroma 目录
    chroma = PersistentClient(path=DB_NAME)
    # 若同名 collection 已存在则删除，保证本次全量重建
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    # 取出每块的 page_content 作为 embedding 输入
    texts = [chunk.page_content for chunk in chunks]
    # 一次 API 调用批量算向量；.data 是 Embedding 对象列表
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    # 只要纯 float 向量
    vectors = [e.embedding for e in emb]

    # 新建空 collection
    collection = chroma.get_or_create_collection(collection_name)

    # id 用 "0","1",... 字符串
    ids = [str(i) for i in range(len(chunks))]
    # 元数据与文本对齐
    metas = [chunk.metadata for chunk in chunks]

    # 四元组一并写入：id / embedding / document / metadata
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    # count() 确认入库条数
    print(f"Vectorstore created with {collection.count()} documents")

# 立刻建库
create_embeddings(chunks)


## 步骤 4：高级 RAG —— 查询改写 + 检索 + 重排序

先把用户问题改写成更利检索的短问，向量召回 Top-K，再用 LLM 按相关性重排，最后带着上下文生成答案。


In [ ]:
# ========== 高级 RAG：重排 / 召回 / 改写 / 生成 ==========

# 重新连接已持久化的 Chroma，拿到 collection 句柄
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)

# 重排结果 schema：按 chunk id（1-based）给出相关性从高到低的顺序
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

# 让 LLM 对召回块重新排序
def rerank(question, chunks):
    # system：说明你是 re-ranker（英文指令保持原样）
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    # user：放入问题 + 逐块正文（CHUNK ID 从 1 起）
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    # 结构化输出 RankOrder
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    # 调试：打印重排后的 id 序列
    print(order)
    # order 里是 1-based 下标，转回 chunks 列表
    return [chunks[i - 1] for i in order]

# 向量召回条数
RETRIEVAL_K = 10

# 只做向量检索，不做重排
def fetch_context_unranked(question):
    # 问题 → embedding 向量
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    # Chroma 近邻查询
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    # documents 与 metadatas 对齐 zip，再包成 Result
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

# 召回后再 rerank
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

# 最终答题用的 system 模板；{context} 运行时 format 填入
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company RoR Inc.
You are chatting with a user about Ruby on Rails.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

# 把检索块拼进 system，并接上 history 与当前 user 问题
def make_rag_messages(question, history, chunks):
    # 每块前注明来源文件，便于模型与用户溯源
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

# 对话感知的查询改写：把闲聊问题收成短检索问句
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    # 注意：下方英文 prompt 保持原样（含 Insurellm 字样，勿改逻辑字符串）
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    # 整段改写指令放在 system role（与原代码一致）
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

# 端到端：改写 → 检索+重排 → 生成答案，并返回上下文块
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    # 先得到更利检索的短问
    query = rewrite_query(question, history)
    print(query)
    # 用改写后的 query 取上下文
    chunks = fetch_context(query)
    # 但生成答案时仍用用户原始 question + history
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

# 可选手动测试（默认注释掉，避免重复烧 token）
# answer_question("如何在 Windows 上安装 Ruby on Rails？", [])


## 步骤 5：Gradio 聊天界面

左侧对话、右侧展示检索到的 Relevant Context；提交消息后走完整 RAG 流水线。


In [ ]:
# ========== Gradio Blocks：双栏聊天 + 检索上下文展示 ==========

# 延迟导入 Gradio（界面相关依赖集中在此格）
import gradio as gr

# 把 Result 列表渲染成带橙色标题的 HTML/Markdown
def format_context(context):
    result = "<h2 style='color: #ff7800;'>Relevant Context</h2>\n\n"
    for doc in context:
        # 每块先显示来源路径
        result += f"<span style='color: #ff7800;'>Source: {doc.metadata['source']}</span>\n\n"
        # 再追加正文
        result += doc.page_content + "\n\n"
    return result

# Chatbot 回调：history 已含最新 user 消息
def chat(history):
    # 最后一条是刚提交的用户内容
    last_message = history[-1]["content"]
    # 之前的轮次作为对话历史
    prior = history[:-1]
    # 调用高级 RAG，同时拿到答案与上下文块
    answer, context = answer_question(last_message, prior)
    # 把助手回复追加进 history
    history.append({"role": "assistant", "content": answer})
    # 返回更新后的对话 + 右侧上下文 HTML
    return history, format_context(context)

# 输入框提交：清空文本框，并把 user 消息塞进 chatbot
def put_message_in_chatbot(message, history):
        return "", history + [{"role": "user", "content": message}]

# Soft 主题 + 指定字体栈
theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

# Blocks：自定义双栏布局（比 ChatInterface 更灵活）
with gr.Blocks(title="Ruby on Rails Expert Assistant", theme=theme) as ui:
    # 页头 Markdown
    gr.Markdown("# 💎 Ruby on Rails Expert Assistant\nAsk me anything about Ruby on Rails!")

    with gr.Row():
        # 左栏：对话
        with gr.Column(scale=1):
            chatbot = gr.Chatbot(
                label="💬 Conversation", height=600, type="messages", show_copy_button=True
            )
            message = gr.Textbox(
                label="Your Question",
                placeholder="Ask anything about Ruby on Rails...",
                show_label=False,
            )

        # 右栏：检索上下文
        with gr.Column(scale=1):
            context_markdown = gr.Markdown(
                label="📄 Retrieved Context",
                value="*Retrieved context will appear here*",
                container=True,
                height=600,
            )
    # 提交链路：先塞 user 消息，再跑 chat 更新答案与上下文
    message.submit(
        put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]
    ).then(chat, inputs=chatbot, outputs=[chatbot, context_markdown])

# 启动并尝试打开浏览器
ui.launch(inbrowser=True)
